In [ ]:
import os, sys, re, ast, time, getpass, subprocess, urllib.request
PORT = 8501

TUNNEL = os.environ.get("TUNNEL", "ngrok")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "streamlit", "pyngrok"], check=True)

import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "deep-translator"], check=True)
print("deep-translator installed successfully!")

deep-translator installed successfully!


In [ ]:
REQUIRED = ["Training.csv", "description.csv", "precautions_df.csv",
            "medications.csv", "diets.csv", "workout_df.csv"]
missing = [f for f in REQUIRED if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(
        f"Missing in the Colab working directory: {missing}. "
        "Upload them using the folder icon on the left, then re-run."
    )

In [ ]:
df_train = pd.read_csv("Training.csv")

df_train = df_train.loc[:, ~df_train.columns.str.contains(r"^Unnamed")]
df_train.columns = [c.strip() for c in df_train.columns]
df_train = df_train.loc[:, ~df_train.columns.str.contains(r"\.\d+$")]

df_train["prognosis"] = df_train["prognosis"].astype(str).str.strip()
df_train = df_train.dropna()
df_train.to_csv("cleaned_training.csv", index=False)

X = df_train.drop("prognosis", axis=1)
y = df_train["prognosis"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_tr, y_tr)
print(f"Hold-out accuracy: {accuracy_score(y_te, model.predict(X_te)) * 100:.2f}%")

model.fit(X, y)
joblib.dump(model, "symptom_model.pkl")
joblib.dump(list(X.columns), "symptom_features.pkl")

profiles = df_train.groupby("prognosis")[list(X.columns)].mean()
joblib.dump(profiles, "symptom_profiles.pkl")


Hold-out accuracy: 100.00%


['symptom_profiles.pkl']

In [ ]:
def norm(s):
    """Normalise disease names so tiny spacing/case differences between CSVs don't break lookups."""
    return re.sub(r"\s+", " ", str(s)).strip().lower()

def to_items(v):
    """Turn a cell into a list of clean strings (handles "['a', 'b']" style cells)."""
    if isinstance(v, str) and v.strip().startswith("["):
        try:
            return [str(i).strip() for i in ast.literal_eval(v) if str(i).strip()]
        except Exception:
            pass
    return [str(v).strip()] if str(v).strip() else []

def build_lookup(filename):
    """disease -> list of items. Handles files with several rows per disease (e.g. workout_df.csv)."""
    data = pd.read_csv(filename)
    data = data.loc[:, ~data.columns.str.contains(r"^Unnamed")]
    key_col = next(c for c in data.columns if c.strip().lower() == "disease")
    val_cols = [c for c in data.columns if c != key_col]
    lookup = {}
    for disease, grp in data.groupby(data[key_col].map(norm)):
        items = []
        for c in val_cols:
            for v in grp[c].dropna():
                items.extend(to_items(v))
        lookup[disease] = list(dict.fromkeys(items))  # de-duplicate, keep order
    return lookup

desc_raw = build_lookup("description.csv")
metadata = {
    "description": {k: " ".join(v) for k, v in desc_raw.items()},
    "precautions": build_lookup("precautions_df.csv"),
    "medications": build_lookup("medications.csv"),
    "diets":       build_lookup("diets.csv"),
    "workout":     build_lookup("workout_df.csv"),
}
joblib.dump(metadata, "medical_metadata.pkl")

for name, lk in metadata.items():
    gap = [d for d in model.classes_ if norm(d) not in lk]
    if gap:
        print(f"[warning] '{name}' has no entry for: {gap}")


[warning] 'description' has no entry for: ['Peptic ulcer diseae']
[warning] 'medications' has no entry for: ['Peptic ulcer diseae']
[warning] 'diets' has no entry for: ['Peptic ulcer diseae']
[warning] 'workout' has no entry for: ['Peptic ulcer diseae']


In [ ]:
app_code = r"""
import re
import numpy as np
import pandas as pd
import joblib
import streamlit as st
from deep_translator import GoogleTranslator

st.set_page_config(page_title="SymptomNav AI", page_icon="🩺", layout="wide")

# ================================================================ LANGUAGES
LANGUAGES = {
    "English": "en",
    "हिन्दी (Hindi)": "hi",
    "ગુજરાતી (Gujarati)": "gu",
    "मराठी (Marathi)": "mr",
    "বাংলা (Bengali)": "bn",
    "தமிழ் (Tamil)": "ta",
    "తెలుగు (Telugu)": "te",
    "ಕನ್ನಡ (Kannada)": "kn",
    "മലയാളം (Malayalam)": "ml",
    "ਪੰਜਾਬੀ (Punjabi)": "pa",
    "اردو (Urdu)": "ur",
}
lang_label = st.sidebar.selectbox("🌐 Language / भाषा / ભાષા", list(LANGUAGES))
LANG = LANGUAGES[lang_label]

# ================================================================ TRANSLATION HELPERS
def _chunks(items, limit=3500):
    cur, size = [], 0
    for it in items:
        if cur and size + len(it) + 1 > limit:
            yield cur
            cur, size = [], 0
        cur.append(it)
        size += len(it) + 1
    if cur:
        yield cur

def _translate_nonempty(texts, lang):
    # Many short texts in ONE request (joined by newlines); falls back to one-by-one if lines mismatch.
    tr = GoogleTranslator(source="en", target=lang)
    out = []
    for chunk in _chunks(texts):
        try:
            parts = tr.translate("\n".join(chunk)).split("\n")
            if len(parts) != len(chunk):
                raise ValueError("line mismatch")
        except Exception:
            parts = []
            for c in chunk:
                try:
                    parts.append(tr.translate(c) or c)
                except Exception:
                    parts.append(c)
        out.extend(parts)
    return out

@st.cache_data(show_spinner=False)
def translate_many(texts, lang):
    clean = [re.sub(r"\s+", " ", str(t)).strip() for t in texts]
    if lang == "en":
        return clean
    idx = [i for i, t in enumerate(clean) if t]
    done = _translate_nonempty([clean[i] for i in idx], lang)
    result = list(clean)
    for i, d in zip(idx, done):
        result[i] = d.strip() or clean[i]
    return result

# ================================================================ INTERFACE TEXT (all translated in one batch)
UI = {
    "caption": "Empowering early health awareness in support of UN SDG 3: Good Health & Well-being",
    "disc_title": "Medical Disclaimer:",
    "disc_body": "This application is for educational purposes only and does not substitute for professional medical advice, diagnosis, or treatment. Always consult a qualified doctor before taking any medication.",
    "mt_note": "This page is machine-translated. Medical terms may not be perfect, so please confirm everything with a doctor.",
    "select_header": "Select Your Current Symptoms",
    "select_label": "Choose all symptoms you are currently experiencing:",
    "run": "Run Health Assessment",
    "none": "Please select at least one symptom to proceed.",
    "predicted": "Predicted Diagnosis",
    "conf": "Assessment Confidence",
    "low": "Confidence is low. Answering the follow-up questions below usually gives a better result.",
    "others": "Other possible conditions (top 3)",
    "col_cond": "Condition",
    "col_prob": "Probability (%)",
    "tab1": "Overview",
    "tab2": "Precautions",
    "tab3": "Medications & Diet",
    "tab4": "Workout & Rest",
    "h_desc": "Disease Description",
    "h_prec": "Recommended Precautions",
    "h_meds": "Common Medications (informational only)",
    "h_diet": "Recommended Diet",
    "h_work": "Exercise & Physical Activity Guidance",
    # ---- feature 1: emergency red flags
    "emerg_title": "EMERGENCY WARNING SIGNS",
    "emerg_body": "Your symptoms may need urgent medical attention. Please call your local emergency number now (India: 112, ambulance: 108) or go to the nearest hospital. Do not wait for the app result.",
    "warn_title": "Please see a doctor soon",
    "warn_body": "Some of the symptoms you selected can be serious. Please visit a doctor or hospital today and do not rely only on this app.",
    "flagged": "Flagged symptoms:",
    "combo_heart": "Chest pain together with breathlessness, sweating or a fast heartbeat can be a sign of a heart attack.",
    "combo_stroke": "Weakness on one side of the body or slurred speech can be a sign of a stroke.",
    "dis_emerg": "The predicted condition can be life-threatening. Please get emergency medical help immediately.",
    "dis_urgent": "The predicted condition needs prompt medical care. Please see a doctor as soon as possible.",
    # ---- feature 2: smart follow-up questions
    "refine_title": "Improve this result (follow-up questions)",
    "refine_q": "These symptoms would help tell the likely conditions apart. Tick the ones you also have:",
    "refine_btn": "Add and update result",
}
_vals = translate_many(tuple(UI.values()), LANG)
T = dict(zip(UI.keys(), _vals))
if LANG != "en" and _vals == [re.sub(r"\s+", " ", v).strip() for v in UI.values()]:
    st.sidebar.warning("Translation service could not be reached, so the page is shown in English. Check the Colab internet connection and try again.")

# ================================================================ HEADER
st.title("🩺 SymptomNav AI")
st.caption(T["caption"])
st.warning(f"⚠️ **{T['disc_title']}** {T['disc_body']}")
if LANG != "en":
    st.info(f"🌐 {T['mt_note']}")

# ================================================================ LOAD MODEL
def norm(s):
    return re.sub(r"\s+", " ", str(s)).strip().lower()

@st.cache_resource
def load_assets():
    return (joblib.load("symptom_model.pkl"), joblib.load("symptom_features.pkl"),
            joblib.load("medical_metadata.pkl"), joblib.load("symptom_profiles.pkl"))

try:
    model, feature_list, metadata, profiles = load_assets()
except Exception as e:
    st.error(f"Could not load model files. Run the training part of the notebook first. Details: {e}")
    st.stop()

pretty = {re.sub(r"\s+", " ", s.replace("_", " ")).strip().title(): s for s in feature_list}
raw_to_pretty = {v: k for k, v in pretty.items()}
english_names = sorted(pretty.keys())
shown = dict(zip(english_names, translate_many(tuple(english_names), LANG)))   # cached per language

# ================================================================ FEATURE 1: EMERGENCY RED FLAGS (rule-based, independent of the ML model)
EMERGENCY_SYMPTOMS = {"coma", "altered_sensorium", "weakness_of_one_body_side",
                      "slurred_speech", "blood_in_sputum", "stomach_bleeding"}
WARNING_SYMPTOMS = {"chest_pain", "breathlessness", "bloody_stool", "stiff_neck",
                    "visual_disturbances", "loss_of_balance"}
HEART_COMBO_WITH = {"breathlessness", "sweating", "fast_heart_rate", "palpitations"}
STROKE_SIGNS = {"weakness_of_one_body_side", "slurred_speech"}
EMERGENCY_DISEASES = {"heart attack", "paralysis (brain hemorrhage)"}
URGENT_DISEASES = {"hypoglycemia", "dengue", "malaria", "typhoid", "pneumonia", "tuberculosis",
                   "aids", "jaundice", "hepatitis b", "hepatitis c", "hepatitis d", "hepatitis e",
                   "alcoholic hepatitis"}

def triage(raw):
    # raw = set of raw feature names. Returns (level, flagged_raw_list, reason_keys). 0=ok 1=see doctor 2=emergency
    emergency = set(raw & EMERGENCY_SYMPTOMS)
    reasons = []
    if "chest_pain" in raw and raw & HEART_COMBO_WITH:
        emergency |= {"chest_pain"} | (raw & HEART_COMBO_WITH)
        reasons.append("combo_heart")
    if raw & STROKE_SIGNS:
        reasons.append("combo_stroke")
    warning = (raw & WARNING_SYMPTOMS) - emergency
    if emergency:
        return 2, sorted(emergency), reasons
    if warning:
        return 1, sorted(warning), reasons
    return 0, [], []

def show_triage(raw):
    level, flagged, reasons = triage(raw)
    if level == 0:
        return
    title, body = (T["emerg_title"], T["emerg_body"]) if level == 2 else (T["warn_title"], T["warn_body"])
    msg = f"{'🚨' if level == 2 else '⚠️'} **{title}**\n\n{body}"
    if reasons:
        msg += "\n\n" + "\n".join(f"- {T[r]}" for r in reasons)
    names = ", ".join(shown[raw_to_pretty[r]] for r in flagged if r in raw_to_pretty)
    if names:
        msg += f"\n\n**{T['flagged']}** {names}"
    (st.error if level == 2 else st.warning)(msg)

# ================================================================ FEATURE 2: SMART FOLLOW-UP QUESTIONS
def suggest_followups(top, selected_raw, k=5):
    # Pick the unselected symptoms that best separate the top candidate diseases.
    # Score = probability-weighted variance of P(symptom | disease) across the candidates.
    cand = [d for d, _ in top if d in profiles.index]
    if len(cand) < 2:
        return []
    w = np.array([p for d, p in top if d in profiles.index], dtype=float)
    w = w / w.sum()
    prof = profiles.loc[cand].drop(columns=[c for c in selected_raw if c in profiles.columns])
    vals = prof.values
    mean = (vals * w[:, None]).sum(axis=0)
    score = (w[:, None] * (vals - mean) ** 2).sum(axis=0)
    order = np.argsort(score)[::-1][:k]
    return [(prof.columns[i], float(score[i])) for i in order if score[i] > 1e-3]

# ================================================================ ASSESSMENT LOGIC (used by both buttons)
def run_assessment():
    sel = st.session_state.get("sel", [])
    st.session_state["empty_warn"] = not sel
    if not sel:
        st.session_state.pop("result", None)
        return
    vec = np.zeros(len(feature_list))
    for name in sel:
        vec[feature_list.index(pretty[name])] = 1
    x = pd.DataFrame([vec], columns=feature_list)
    proba = model.predict_proba(x)[0]
    order = np.argsort(proba)[::-1][:3]
    st.session_state["result"] = {
        "top": [(str(model.classes_[i]), float(proba[i] * 100)) for i in order],
        "selected": list(sel),
    }

def refine_assessment():
    ticked = [n for n in english_names if st.session_state.get(f"fu_{n}")]
    for n in ticked:
        st.session_state[f"fu_{n}"] = False
    st.session_state["sel"] = sorted(set(st.session_state.get("sel", [])) | set(ticked))
    run_assessment()

# ================================================================ SYMPTOM PICKER
st.subheader(f"📋 {T['select_header']}")
selected = st.multiselect(
    T["select_label"],
    options=english_names,                    # values stay English so the model input is unchanged
    format_func=lambda n: shown.get(n, n),    # what the user SEES is in their language
    key="sel",
)

# Red-flag banner appears instantly, before any button is pressed
show_triage({pretty[n] for n in selected})

st.button(f"🔍 {T['run']}", type="primary", on_click=run_assessment)
if st.session_state.get("empty_warn"):
    st.info(T["none"])

# ================================================================ RESULTS
res = st.session_state.get("result")
if res:
    prediction, confidence = res["top"][0]
    key = norm(prediction)

    desc = [metadata["description"].get(key, "") or "No description available."]
    precs = [p.capitalize() for p in metadata["precautions"].get(key, [])] or ["Consult a doctor."]
    meds = metadata["medications"].get(key, []) or ["Consult a medical professional."]
    diet = metadata["diets"].get(key, []) or ["Maintain a balanced diet."]
    work = metadata["workout"].get(key, []) or ["Adequate rest recommended."]
    others = [d for d, _ in res["top"]]

    blocks = [desc, precs, meds, diet, work, others]
    flat_tr = translate_many(tuple(t for b in blocks for t in b), LANG)
    out, pos = [], 0
    for b in blocks:
        out.append(flat_tr[pos:pos + len(b)])
        pos += len(b)
    desc_t, precs_t, meds_t, diet_t, work_t, others_t = out

    def bullets(items):
        return "\n".join(f"- {i}" for i in items)

    disease_shown = others_t[0] if (LANG == "en" or others_t[0] == prediction) else f"{others_t[0]} ({prediction})"

    st.markdown("---")
    if key in EMERGENCY_DISEASES:
        st.error(f"🚨 **{T['emerg_title']}**\n\n{T['dis_emerg']}\n\n{T['emerg_body']}")
    elif key in URGENT_DISEASES:
        st.warning(f"⚠️ {T['dis_urgent']}")

    st.success(f"### {T['predicted']}: **{disease_shown}**")
    st.metric(label=T["conf"], value=f"{confidence:.1f}%")
    if confidence < 40:
        st.info(T["low"])

    with st.expander(T["others"]):
        st.dataframe(
            pd.DataFrame({T["col_cond"]: others_t, T["col_prob"]: [round(p, 1) for _, p in res["top"]]}),
            hide_index=True,
        )

    # ---- Feature 2 UI
    followups = suggest_followups(res["top"], {pretty[n] for n in res["selected"]})
    if followups:
        with st.expander(f"🧠 {T['refine_title']}", expanded=confidence < 70):
            st.write(T["refine_q"])
            cols = st.columns(2)
            for i, (raw, _) in enumerate(followups):
                n = raw_to_pretty[raw]
                cols[i % 2].checkbox(shown.get(n, n), key=f"fu_{n}")
            st.button(f"➕ {T['refine_btn']}", on_click=refine_assessment, key="refine_btn")

    tab1, tab2, tab3, tab4 = st.tabs(
        [f"📖 {T['tab1']}", f"🛡️ {T['tab2']}", f"💊 {T['tab3']}", f"🏋️ {T['tab4']}"]
    )
    with tab1:
        st.write(f"#### {T['h_desc']}")
        st.write(desc_t[0])
    with tab2:
        st.write(f"#### {T['h_prec']}")
        st.markdown(bullets(precs_t))
    with tab3:
        col_a, col_b = st.columns(2)
        with col_a:
            st.write(f"#### {T['h_meds']}")
            st.info(bullets(meds_t))
        with col_b:
            st.write(f"#### {T['h_diet']}")
            st.success(bullets(diet_t))
    with tab4:
        st.write(f"#### {T['h_work']}")
        st.warning(bullets(work_t))
"""
with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

In [ ]:
subprocess.run('pkill -f "[s]treamlit run"', shell=True)
subprocess.run('pkill -f "[c]loudflared tunnel"', shell=True)
time.sleep(2)

st_log = open("streamlit.log", "w")
st_proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py",
     "--server.port", str(PORT),
     "--server.address", "0.0.0.0",
     "--server.headless", "true",
     "--server.enableCORS", "false",
     "--server.enableXsrfProtection", "false",
     "--browser.gatherUsageStats", "false"],
    stdout=st_log, stderr=subprocess.STDOUT,
)

# Wait until Streamlit is really up
ready = False
for _ in range(60):
    if st_proc.poll() is not None:
        break
    try:
        with urllib.request.urlopen(f"http://localhost:{PORT}/_stcore/health", timeout=2) as r:
            if r.read().decode().strip() == "ok":
                ready = True
                break
    except Exception:
        pass
    time.sleep(1)

if not ready:
    print(open("streamlit.log").read())
    raise RuntimeError("Streamlit failed to start - see the log above.")
print("Streamlit has successfully restarted!")

Streamlit has successfully restarted!


In [ ]:
public_url = None

try:
    if TUNNEL == "ngrok":
        from pyngrok import ngrok
        token = None
        try:
            from google.colab import userdata          # Colab "Secrets" (key icon on the left)
            token = userdata.get("NGROK_AUTHTOKEN")
        except Exception:
            token = os.environ.get("NGROK_AUTHTOKEN")
        if not token:
            token = getpass.getpass("Paste your ngrok authtoken (free at dashboard.ngrok.com): ").strip()
        ngrok.set_auth_token(token)
        for t in ngrok.get_tunnels():
            ngrok.disconnect(t.public_url)
        public_url = ngrok.connect(PORT, "http").public_url
except Exception as e:
    print(f"ngrok failed: {e}\nFalling back to cloudflared...")
    TUNNEL = "cloudflared"

if TUNNEL == "cloudflared":
    if not os.path.exists("cloudflared"):
        subprocess.run(
            ["wget", "-q", "-O", "cloudflared",
             "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
            check=True)
        os.chmod("cloudflared", 0o755)
    cf_log = open("cloudflared.log", "w")
    subprocess.Popen(["./cloudflared", "tunnel", "--url", f"http://localhost:{PORT}", "--no-autoupdate"],
                     stdout=cf_log, stderr=subprocess.STDOUT)
    for _ in range(40):
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("cloudflared.log").read())
        if m:
            public_url = m.group(0)
            break
        time.sleep(1)
    if not public_url:
        print(open("cloudflared.log").read())
        raise RuntimeError("cloudflared did not give a URL - see the log above.")

if public_url:
    print("=" * 60)
    print(f"URL: {public_url}")
    if TUNNEL == "ngrok":
        print("If ngrok shows a warning page, click 'Visit Site' once.")
    print("Open it in a NEW browser tab (don't reuse an old tab).")
    print("=" * 60)

Paste your ngrok authtoken (free at dashboard.ngrok.com): ··········
URL: https://quizzical-margin-napkin.ngrok-free.dev
If ngrok shows a warning page, click 'Visit Site' once.
Open it in a NEW browser tab (don't reuse an old tab).


ngrok authtoken: 3KRwucOcAGsowZ4YC4PJX7Mb6Qx_65FBoxhA9xqoKaGAnqXVV
Kaggle Dataset: https://www.kaggle.com/datasets/sibamsamanta07/disease-symptom-prediction-and-treatment-dataset